# 冻结分析瓶颈与等价优化

仅复核既有 Qlib Recorder 的分析计算；没有新拟合、参数搜索、市场回测或策略晋升。假设：逐抽样的 SciPy 排名/相关系数调用是热点，可在相同随机索引上分批计算，减少 Python 调用与重复排序。

输入固定为 `frozen_cash_information_20261003` 已完成记录。比较归档源码、归档结果与当前实现；所有 4 × 2,000 × 9 个统计值、区间、NaN 位置和索引 hash 必须一致（数值容差 `atol=1e-12, rtol=1e-12`，用于浮点归约顺序差异）。计时不含导入、文件加载或 profiler；3 次交错测量取中位数。此卡属于工程验证，不重开历史研究门。

隔离 worktree 缺少旧 Recorder 的未跟踪运行元数据时，显式设置 `SOXL_CANONICAL_EVIDENCE_ROOT` 指向完整主仓；输入仍由原 Qlib Recorder 读取并逐文件校验 hash。当前实现始终从执行仓库的 `src/` 加载。

In [1]:
import os, sys, json, hashlib, types, time, cProfile, pstats, platform
from pathlib import Path
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "SPEC.md").exists())
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import scipy
import qlib
from qlib.data import D
from qlib.workflow import R
from src.analysis import frozen_cash_information as optimized
NAME = "frozen_analysis_performance_20261008"
CARD = ROOT / "research" / NAME
EVIDENCE_ROOT = Path(os.environ.get("SOXL_CANONICAL_EVIDENCE_ROOT", str(ROOT))).resolve()
SOURCE = EVIDENCE_ROOT / "research/frozen_cash_information_20261003"
ATOL = RTOL = 1e-12
sha = lambda path: hashlib.sha256(Path(path).read_bytes()).hexdigest()
source_summary = json.loads((SOURCE / "summary.json").read_text())
reference = json.loads((SOURCE / "recorder_reference.json").read_text())
provider = EVIDENCE_ROOT / source_summary["provider_uri"]
assert provider.is_dir(), "blocked: source Qlib provider missing"
qlib.init(provider_uri=str(provider), region="us", kernels=1)
calendar = pd.DatetimeIndex(D.calendar(freq="day"))
assert str(calendar.max().date()) == source_summary["source"]["as_of"]
source_uri = str((SOURCE / "qlib_recorder").resolve())
with R.uri_context(source_uri):
    source_recorder = R.get_recorder(experiment_name=source_summary["experiment"], recorder_id=source_summary["recorder_id"])
    assert source_recorder.client.get_run(source_recorder.id).info.status == "FINISHED"
    source_dir = Path(source_recorder.get_local_dir()) / "artifacts"
    artifact_names = ["inputs/diagnostic_rows.pkl", "analysis/bootstrap_draws.pkl", "analysis/tables.pkl"]
    bindings = {}
    for artifact in artifact_names:
        assert sha(source_dir / artifact) == reference["artifacts"][artifact]["sha256"]
        bindings[artifact] = reference["artifacts"][artifact]
    rows = source_recorder.load_object("inputs/diagnostic_rows.pkl")
    saved_draws = source_recorder.load_object("analysis/bootstrap_draws.pkl")
    saved_tables = source_recorder.load_object("analysis/tables.pkl")
assert rows.index.isin(calendar).all()
source_path = source_dir / "source/src/analysis/frozen_cash_information.py"
assert sha(source_path) == source_summary["source_sha256"]["src/analysis/frozen_cash_information.py"]
baseline = types.ModuleType("archived_frozen_cash_information")
exec(compile(source_path.read_text(), str(source_path), "exec"), baseline.__dict__)
plan = json.loads((SOURCE / "diagnostic_plan.json").read_text())
sampler_path = ROOT / "src/objective/relative_wealth.py"
assert sha(sampler_path) == "3d1c58c381687de928c06cfa9ce2715bcb2d6bc312de6e2a9a74f96ebe279549"
print({"source_recorder": source_recorder.id, "source_rows": len(rows), "calendar_sessions": len(calendar), "as_of": source_summary["source"]["as_of"]})

2026/10/08 10:50:10 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


[2325:MainThread](2026-10-08 10:50:10,129) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[2325:MainThread](2026-10-08 10:50:10,130) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[2325:MainThread](2026-10-08 10:50:10,130) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/published_futu_affine_20260925_v1')}


{'source_recorder': 'ef3da36b5e6a4f4297c2bf76a6a19afd', 'source_rows': 3956, 'calendar_sessions': 4162, 'as_of': '2026-09-25'}


## 原始结果与优化结果的逐项核验

使用已归档分析结果作为第三方对照，避免只验证两个新计算互相一致。保留所有缺失/未定义值及完整费用诊断。固定去均值仍在重采样前完成。

In [2]:
def assert_table_equal(left, right):
    pd.testing.assert_frame_equal(left, right, check_exact=False, atol=ATOL, rtol=RTOL)
    assert left.isna().equals(right.isna())

old_result = baseline.analyze_information(rows, plan)
new_result = optimized.analyze_information(rows, plan)
parity_rows = []
for key, saved in saved_draws.items():
    old = old_result["bootstrap_draws"][key]
    new = new_result["bootstrap_draws"][key]
    for name in set(saved) - {"statistics"}:
        assert old[name] == new[name] == saved[name], (key, name)
    assert_table_equal(old["statistics"], saved["statistics"])
    assert_table_equal(new["statistics"], saved["statistics"])
    delta = np.abs(new["statistics"].to_numpy() - saved["statistics"].to_numpy())
    parity_rows.append({"window": key[0], "mean_block": key[1], "rows": saved["indices_shape"][1],
                        "draws": len(new["statistics"]), "statistics": len(new["statistics"].columns),
                        "max_absolute_difference": float(np.nanmax(delta)), "nan_mask_equal": True,
                        "indices_sha256": saved["indices_sha256"]})
for name in ("point_metrics", "calibration", "event_metrics", "bootstrap_summary"):
    assert_table_equal(old_result[name], saved_tables[name + ".csv"])
    assert_table_equal(new_result[name], saved_tables[name + ".csv"])
assert old_result["audit"] == new_result["audit"]
# Preserve interval validity and all sign-based downstream decisions, as well as values.
bs0, bs1 = old_result["bootstrap_summary"], new_result["bootstrap_summary"]
for name in ("official_interval_defined", "undefined_draws", "defined_draws"):
    pd.testing.assert_series_equal(bs0[name], bs1[name])
for name in ("lower_2_5", "median_50", "upper_97_5"):
    np.testing.assert_array_equal(np.sign(bs0[name].to_numpy(float)), np.sign(bs1[name].to_numpy(float)))
parity = pd.DataFrame(parity_rows)
parity

,window,mean_block,rows,draws,statistics,max_absolute_difference,nan_mask_equal,indices_sha256
0,development_2016_2022,20,1762,2000,9,8.526513e-14,True,14f92ee767b831ca9198e4f0bf68e0b977880e3f7e26ba...
1,development_2016_2022,60,1762,2000,9,5.684342e-14,True,c7e57126a3a2c8a0cbc8cb19f1aff3c0c5a97a733051c0...
2,reused_2023_asof,20,934,2000,9,5.684342e-14,True,0c01fbc9e37507235ada405cd4798acd445f5443065193...
3,reused_2023_asof,60,934,2000,9,8.526513e-14,True,41c0f7a895e0964639d2157cd8346a10fe7ca44eb323d0...


## 热点与计时

Profiler 单独运行，只用于定位耗时；优化倍数来自不启用 profiler 的相同进程、交错重复测量。开发期与复用期分别报告；不会将该分析内核的改进说成训练或整个平台的加速。

In [3]:
sample = rows.loc[rows.score_available & rows.index.year.isin(range(2016, 2023))]
args = (sample.cash_advantage_1d.to_numpy(float), sample.cash_advantage_prediction.to_numpy(float),
        sample.prior_mean_prediction.to_numpy(float), sample.index.year.to_numpy())
profiles, profile_rows = {}, []
for label, module in (("baseline", baseline), ("optimized", optimized)):
    profiler = cProfile.Profile()
    profiler.runcall(module.bootstrap_window, *args, mean_block=20, resamples=2000, seed=20261003)
    stats = pstats.Stats(profiler)
    profiles[label] = {"total_seconds": stats.total_tt, "total_calls": stats.total_calls}
    for (filename, line, function), (primitive, calls, internal, cumulative, _) in stats.stats.items():
        if function in ("correlation", "rankdata", "stationary_block_bootstrap_indices", "bootstrap_window"):
            profile_rows.append({"implementation": label, "function": function, "calls": calls,
                                 "internal_seconds": internal, "cumulative_seconds": cumulative,
                                 "fraction_of_profile_total": cumulative / stats.total_tt})
profile_table = pd.DataFrame(profile_rows)
profile_table

,implementation,function,calls,internal_seconds,cumulative_seconds,fraction_of_profile_total
0,baseline,rankdata,8000,0.015289,1.020069,0.742428
1,baseline,stationary_block_bootstrap_indices,1,0.030431,0.033015,0.024029
2,baseline,bootstrap_window,1,0.060106,0.802335,0.583956
3,baseline,correlation,8000,0.058607,1.192461,0.867899
4,optimized,stationary_block_bootstrap_indices,1,0.030206,0.032729,0.172921
5,optimized,bootstrap_window,1,0.033063,0.189228,0.999765


In [4]:
timing_rows = []
for key, saved in saved_draws.items():
    years = range(2016, 2023) if key[0] == "development_2016_2022" else range(2023, 2027)
    sample = rows.loc[rows.score_available & rows.index.year.isin(years)]
    args = (sample.cash_advantage_1d.to_numpy(float), sample.cash_advantage_prediction.to_numpy(float),
            sample.prior_mean_prediction.to_numpy(float), sample.index.year.to_numpy())
    for repeat in range(3):
        order = (("baseline", baseline), ("optimized", optimized))
        if repeat % 2: order = order[::-1]
        for label, module in order:
            started = time.perf_counter()
            measured = module.bootstrap_window(*args, mean_block=key[1], resamples=2000, seed=20261003)
            elapsed = time.perf_counter() - started
            assert measured["indices_sha256"] == saved["indices_sha256"]
            timing_rows.append({"window": key[0], "mean_block": key[1], "implementation": label,
                                "repeat": repeat, "seconds": elapsed})
# Same full analysis interface, including all small tables and four bootstrap cases.
for repeat in range(3):
    order = (("baseline", baseline), ("optimized", optimized))
    if repeat % 2: order = order[::-1]
    for label, module in order:
        started = time.perf_counter()
        measured = module.analyze_information(rows, plan)
        elapsed = time.perf_counter() - started
        assert measured["audit"] == new_result["audit"]
        timing_rows.append({"window": "full_analysis", "mean_block": 0, "implementation": label,
                            "repeat": repeat, "seconds": elapsed})
timings = pd.DataFrame(timing_rows)
medians = timings.groupby(["window", "mean_block", "implementation"]).seconds.median().unstack().reset_index()
medians["speedup"] = medians.baseline / medians.optimized
medians["reduction_pct"] = 100 * (1 - medians.optimized / medians.baseline)
medians

implementation,window,mean_block,baseline,optimized,speedup,reduction_pct
0,development_2016_2022,20,1.047505,0.183759,5.700426,82.457453
1,development_2016_2022,60,1.038692,0.186302,5.575304,82.063760
2,full_analysis,0,3.460877,0.593035,5.835875,82.864609
3,reused_2023_asof,20,0.667636,0.102554,6.510086,84.639219
4,reused_2023_asof,60,0.662239,0.104070,6.363379,84.285080


## Recorder 归档与科学边界

新工程记录保存性能测量、完整等价统计与源码身份；原 Recorder、执行 seal、科研结论均不改。已有实验的旧 seal 会正确拒绝当前源码，复现原结果仍使用归档代码。未来新研究必须重新绑定实际源码，不能替换历史 hash。

In [5]:
full = medians.loc[medians.window.eq("full_analysis")].iloc[0]
summary = {
    "experiment": NAME, "record_type": "frozen_analysis_engineering_verification",
    "source_experiment": source_summary["experiment"], "source_recorder_id": source_recorder.id,
    "provider_uri": source_summary["provider_uri"], "region": "us", "market": "all",
    "as_of": source_summary["source"]["as_of"], "source_rows": len(rows),
    "source_artifact_bindings": bindings,
    "baseline_source_sha256": sha(source_path), "sampler_sha256": sha(sampler_path),
    "implementation_sha256": sha(ROOT / "src/analysis/frozen_cash_information.py"),
    "parity": {"passed": True, "statistic_cells": 72000, "bootstrap_interval_rows": len(bs1),
               "atol": ATOL, "rtol": RTOL, "max_absolute_difference": float(parity.max_absolute_difference.max()),
               "indices_hashes_equal": True, "nan_masks_equal": True, "interval_status_and_signs_equal": True},
    "timing": {"repetitions": 3, "method": "alternating paired runs, median; excludes imports, input I/O, profiling",
               "full_analysis_baseline_seconds": float(full.baseline),
               "full_analysis_optimized_seconds": float(full.optimized), "full_analysis_speedup": float(full.speedup)},
    "environment": {"python": platform.python_version(), "machine": platform.machine(), "platform": platform.platform(),
                    "numpy": np.__version__, "pandas": pd.__version__, "scipy": scipy.__version__, "qlib": qlib.__version__},
    "market_model_fits": 0, "market_backtests": 0, "production_change": False,
    "promotion_decision": "not_promoted_engineering_only", "stable_excess_achieved": False,
    "workflow_config": "workflow.yaml",
    "artifact_policy": {"policy": "slim_card_recorder_first", "workflow_config": "workflow.yaml",
        "recorder_only_artifacts": ["analysis/bootstrap_draws.pkl", "analysis/timing_samples.pkl", "analysis/tables.pkl"]},
    "sidecars": ["timing_summary.csv", "parity_summary.csv", "profile_summary.csv", "conclusion.md"],
    "qlib_docs_reference": ["https://qlib.readthedocs.io/en/latest/component/data.html",
                            "https://qlib.readthedocs.io/en/latest/component/recorder.html"],
    "limitations": ["Frozen reused history, no new alpha evidence", "Timing is specific to this host and workload",
                    "Sampler pinned separately; old source manifest did not archive its full dependency closure",
                    "Historical execution seals and result cards remain unchanged"]}
uri = str((CARD / "qlib_recorder").resolve())
with R.start(experiment_name=NAME, uri=uri):
    recorder = R.get_recorder()
    summary.update(recorder_id=recorder.id, experiment_id=recorder.experiment_id,
                   recorder_uri=str((CARD / "qlib_recorder").relative_to(ROOT)),
                   recorder_relative_dir=str(Path(recorder.get_local_dir()).relative_to(CARD / "qlib_recorder")))
    R.log_params(source_recorder_id=source_recorder.id, baseline_source_sha256=sha(source_path),
                 implementation_sha256=summary["implementation_sha256"], sampler_sha256=sha(sampler_path))
    R.log_metrics(full_analysis_speedup=float(full.speedup), parity_statistic_cells=72000,
                  max_absolute_difference=summary["parity"]["max_absolute_difference"])
    R.save_objects(artifact_path="analysis", **{"bootstrap_draws.pkl": new_result["bootstrap_draws"],
        "tables.pkl": {name: new_result[name] for name in ("point_metrics", "calibration", "event_metrics", "bootstrap_summary")},
        "timing_samples.pkl": timings, "summary.pkl": summary})
    recorder.log_artifact(str(source_path), artifact_path="source/baseline")
    recorder.log_artifact(str(ROOT / "src/analysis/frozen_cash_information.py"), artifact_path="source/optimized")
    recorder.log_artifact(str(sampler_path), artifact_path="source/sampler")
    medians.to_csv(CARD / "timing_summary.csv", index=False)
    parity.to_csv(CARD / "parity_summary.csv", index=False)
    profile_table.to_csv(CARD / "profile_summary.csv", index=False)
    for filename in ("timing_summary.csv", "parity_summary.csv", "profile_summary.csv", "workflow.yaml"):
        recorder.log_artifact(str(CARD / filename), artifact_path="card")
(CARD / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2, allow_nan=False) + "\n")
conclusion = f"""# 冻结分析优化结论

完成既有现金预测的 Qlib Recorder 分析重放：开发期 2016–2022（1762 行）、复用期 2023–2026-09-23（934 行），各 20/60 日块、2000 次抽样。没有模型拟合或市场回测。

四组共 72,000 个统计值、36 行区间、全部抽样索引 hash、NaN 位置及区间符号通过等价核验；最大绝对差 {summary['parity']['max_absolute_difference']:.3g}，声明浮点容差 atol=rtol=1e-12。全部小表与归档 Recorder 结果一致。

完整 analyze_information 中位耗时从 {full.baseline:.4f}s 降至 {full.optimized:.4f}s，约 {full.speedup:.2f} 倍；各窗口见 timing_summary.csv，重复样本仅存 Recorder。该结果仅适用于当前主机上的冻结分析，不代表训练或整个平台加速。逐年/窗口科学结论保持原值；未新增事后 regime 筛选。

不晋升。信号有效性与历史时点数据不足的瓶颈仍在：广度路线已失败，财报预测误差未改善且缺少历史版本，旧规则未通过风险门。运行更快不会产生新的稳定超额证据。

原 Recorder、源码归档与执行 seal 保持不变；当前优化源码须由新实验另行绑定，不能重签旧结论。下一步应先取得可信历史时点数据或新的连续样本；工程上仅在新 profile 指向其他热点时继续优化。

Qlib Recorder：{recorder.id}。输入源：{source_recorder.id}；provider：{summary['provider_uri']}，as_of={summary['as_of']}。
"""
(CARD / "conclusion.md").write_text(conclusion)
print(conclusion)

[2325:MainThread](2026-10-08 10:50:39,944) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name frozen_analysis_performance_20261008.


[2325:MainThread](2026-10-08 10:50:39,946) INFO - qlib.workflow - [exp.py:258] - Experiment 466430838156371904 starts running ...


[2325:MainThread](2026-10-08 10:50:40,009) INFO - qlib.workflow - [recorder.py:345] - Recorder a200119a467d4cb197b5dded98a3c28a starts running under Experiment 466430838156371904 ...


[2325:MainThread](2026-10-08 10:50:40,091) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | waiting `async_log` Done


# 冻结分析优化结论

完成既有现金预测的 Qlib Recorder 分析重放：开发期 2016–2022（1762 行）、复用期 2023–2026-09-23（934 行），各 20/60 日块、2000 次抽样。没有模型拟合或市场回测。

四组共 72,000 个统计值、36 行区间、全部抽样索引 hash、NaN 位置及区间符号通过等价核验；最大绝对差 8.53e-14，声明浮点容差 atol=rtol=1e-12。全部小表与归档 Recorder 结果一致。

完整 analyze_information 中位耗时从 3.4609s 降至 0.5930s，约 5.84 倍；各窗口见 timing_summary.csv，重复样本仅存 Recorder。该结果仅适用于当前主机上的冻结分析，不代表训练或整个平台加速。逐年/窗口科学结论保持原值；未新增事后 regime 筛选。

不晋升。信号有效性与历史时点数据不足的瓶颈仍在：广度路线已失败，财报预测误差未改善且缺少历史版本，旧规则未通过风险门。运行更快不会产生新的稳定超额证据。

原 Recorder、源码归档与执行 seal 保持不变；当前优化源码须由新实验另行绑定，不能重签旧结论。下一步应先取得可信历史时点数据或新的连续样本；工程上仅在新 profile 指向其他热点时继续优化。

Qlib Recorder：a200119a467d4cb197b5dded98a3c28a。输入源：ef3da36b5e6a4f4297c2bf76a6a19afd；provider：data/qlib/published_futu_affine_20260925_v1，as_of=2026-09-25。



# 冻结分析优化结论

完成既有现金预测的 Qlib Recorder 分析重放：开发期 2016–2022（1762 行）、复用期 2023–2026-09-23（934 行），各 20/60 日块、2000 次抽样。没有模型拟合或市场回测。

四组共 72,000 个统计值、36 行区间、全部抽样索引 hash、NaN 位置及区间符号通过等价核验；最大绝对差 8.53e-14，声明浮点容差 atol=rtol=1e-12。全部小表与归档 Recorder 结果一致。

完整 analyze_information 中位耗时从 3.4609s 降至 0.5930s，约 5.84 倍；各窗口见 timing_summary.csv，重复样本仅存 Recorder。该结果仅适用于当前主机上的冻结分析，不代表训练或整个平台加速。逐年/窗口科学结论保持原值；未新增事后 regime 筛选。

不晋升。信号有效性与历史时点数据不足的瓶颈仍在：广度路线已失败，财报预测误差未改善且缺少历史版本，旧规则未通过风险门。运行更快不会产生新的稳定超额证据。

原 Recorder、源码归档与执行 seal 保持不变；当前优化源码须由新实验另行绑定，不能重签旧结论。下一步应先取得可信历史时点数据或新的连续样本；工程上仅在新 profile 指向其他热点时继续优化。

Qlib Recorder：a200119a467d4cb197b5dded98a3c28a。输入源：ef3da36b5e6a4f4297c2bf76a6a19afd；provider：data/qlib/published_futu_affine_20260925_v1，as_of=2026-09-25。

归档说明：本次在隔离 worktree 执行，旧输入从完整主仓的 Qlib Recorder 只读加载并校验 hash。执行后将本结论同步到 notebook，补存最终 notebook、结论、测试源码及精确文件清单。仅新 Recorder 的 artifact URI 元数据映射至合并后的主仓位置，详情见 `relocation.json`；归档文件的用途、来源、日期范围及 SHA-256 见 `verification_manifest.json`。
